In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# 05 \u2014 Convergence Behavior"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Objectives\n",
        "- Study likelihood histories across random starts.\n",
        "- Demonstrate local optima and poor initialization.\n",
        "- Show why covariance regularization matters."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Mathematical Background\n",
        "$\\Sigma_k\\leftarrow\\Sigma_k+\\epsilon I$ prevents nearly singular covariance estimates when a component collapses around too few points."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Implementation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import sys\n",
        "from pathlib import Path\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "\n",
        "PROJECT = Path.cwd()\n",
        "if PROJECT.name == 'notebooks':\n",
        "    PROJECT = PROJECT.parent\n",
        "FIGURES = PROJECT / 'figures'\n",
        "FIGURES.mkdir(exist_ok=True)\n",
        "if str(PROJECT) not in sys.path:\n",
        "    sys.path.insert(0, str(PROJECT))\n",
        "\n",
        "plt.style.use('seaborn-v0_8-whitegrid')\n",
        "COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']\n",
        "RNG = np.random.default_rng(7)\n",
        "\n",
        "def savefig(name):\n",
        "    plt.tight_layout()\n",
        "    plt.savefig(FIGURES / name, dpi=180, bbox_inches='tight', transparent=True)\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Experiments"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from src.gaussian import sample_mixture\n",
        "from src.gmm import GaussianMixtureModel\n",
        "X,_=sample_mixture(np.array([.5,.3,.2]), np.array([[-2,0],[1,2],[2,-1.5]]), np.array([[[.6,.1],[.1,.5]],[[.7,.35],[.35,.7]],[[.4,0],[0,.4]]]), 900, random_state=RNG)\n",
        "plt.figure(figsize=(7,4))\n",
        "finals=[]\n",
        "for seed in range(6):\n",
        "    model=GaussianMixtureModel(3, max_iter=60, random_state=seed, reg_covar=1e-5).fit(X)\n",
        "    finals.append(model.lower_bound_history_[-1])\n",
        "    plt.plot(model.lower_bound_history_, alpha=.75, label=f'seed {seed}')\n",
        "plt.title('How sensitive is EM convergence to initialization?'); plt.xlabel('iteration'); plt.ylabel('log likelihood'); plt.legend(ncol=2, fontsize=8); savefig('convergence_curve.png')"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "fig, axes = plt.subplots(1,2,figsize=(10,4))\n",
        "for ax, k in zip(axes,[2,5]):\n",
        "    model=GaussianMixtureModel(k, max_iter=80, random_state=3, reg_covar=1e-6).fit(X)\n",
        "    ax.scatter(X[:,0],X[:,1],c=model.predict(X),cmap='viridis',s=10,alpha=.55)\n",
        "    ax.set_title(f'K={k}, final LL={model.lower_bound_history_[-1]:.1f}')\n",
        "savefig('local_optimum.png')\n",
        "print('Final likelihoods by seed:', np.round(finals, 2))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Observations\n",
        "- Multiple starts can end at different likelihoods.\n",
        "- Too few components underfit; too many components can chase small groups unless regularized."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Key Takeaways\n",
        "- Monotonic improvement is not the same as global optimality.\n",
        "- Regularization is a practical numerical and statistical safeguard."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Transition to the next notebook\n",
        "Next we use posterior probabilities for supervised-style classification and compare decision boundaries."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "pygments_lexer": "ipython3"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
machine_learning/gmm/notebooks/05_elbo_convergence.ipynb
